# Calibration-constrained NonLinear Uncertainty Analysis - Iterative Ensemble Smoother

As we've seen, First-Order-Second-Moment (FOSM) is quick and insightful.  But FOSM depends on an assumption that the relation between the model and the forecast uncertainty is linear.  But many times the groundwater modeling inverse problems are nonlinear. 

On a more practical standpoint, the underlying theory for FOSM can be hard to explain.  Monte Carlo, however, is straightforward, its computational brute force notwithstanding.  The problem with pure Monte Carlo is the hit-to-miss ratio for conditioning to observations: most parameter sets drawn at random from the prior fit the historical data poorly, so a lot of forward runs are effectively wasted.

The iterative ensemble smoother (IES) is designed to solve this. IES starts a prior ensemble of parameter sets, but then uses linear algebra and approximate gradient information to adjust ensemble so each realisation moves toward fitting the observations. The result is an approximate posterior ensemble that is calibration-constrained (it honours the historical data) and still honors the prior. In this notebook we run IES with `pestpp-ies`.

## The Current Tutorial

In this notebook we will:
1. Run iterative ensemble smoother on the Freyberg model
2. Look at parameter and forecast uncertainty 
3. Look at the effect of prior parameter uncertainty covariance
4. Start thinking of the advantages and disadvantages of linear and nonlinear uncertainty methods

This notebook is going to be computationally tough and may tax your computer. So buckle up. 


### Admin

First the usual admin of preparing folders and constructing the model and PEST datasets.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;

import shutil

import pyemu
import flopy
sys.path.insert(0,"..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10
pyemu.plot_utils.font =10

In [ ]:
# folder containing original model files
org_d = os.path.join('..', '..', 'models', 'monthly_model_files_1lyr_newstress')
# a dir to hold a copy of the org model files
working_dir = os.path.join('freyberg_mf6')
if os.path.exists(working_dir):
    shutil.rmtree(working_dir)
shutil.copytree(org_d,working_dir)
# get executables
hbd.prep_bins(working_dir)
# get dependency folders
# run our convenience functions to prepare the PEST and model folder
hbd.prep_pest(working_dir)
# convenience function that builds a new control file with pilot point parameters for hk.
# NOTE: it also weights the gage-1 observations with hbd.SFR_WEIGHT - the same
# (subjective!) streamflow weight used across part1; see the part1_01 trial-and-error notebook
hbd.add_ppoints(working_dir)


IES uses lots and lots of forward runs so we don't want to make the mistake of burning the silicon for a PEST control file that is not right.  Here we make doubly sure that the control file has the recharge freed (not "fixed" in the PEST control file).  

### Load the `pst` control file

Let's double check what parameters we have in this version of the model using `pyemu` (you can just look in the PEST control file too.).

We have adjustable parameters that control SFR inflow rates, well pumping rates, hydraulic conductivity and recharge rates. Recall that by setting a parameter as "fixed" we are stating that we know it perfectly (should we though...?). Currently fixed parameters include porosity and future recharge.

For the sake of this tutorial, and as we did in the "sensitivity" tutorials, let's set all the parameters free:

In [ ]:
pst_name = "freyberg_pp.pst"
# load the pst
pst = pyemu.Pst(os.path.join(working_dir,pst_name))
#update parameter data
par = pst.parameter_data
#update parameter transform
par["partrans"] = 'log'

In [ ]:
# check if the model runs
pst.control_data.noptmax=0
# rewrite the control file
pst.write(os.path.join(working_dir,pst_name))
# run the model once
pyemu.os_utils.run('pestpp-glm freyberg_pp.pst', cwd=working_dir)

Reload the control file:

In [ ]:
pst = pyemu.Pst(os.path.join(working_dir,'freyberg_pp.pst'))
pst.phi

# The Prior Parameter Ensemble

Just like Monte Carlo, IES starts from an ensemble of parameter sets "drawn" from the prior probability distribution; IES will then iteratively adjust that ensemble to fit the observations. But first we need the prior.

So how do we "draw", or sample, parameters? (Think "draw" as in "drawing a card from a deck"). We need to randomly sample parameter values from a range. This range is defined by the _prior parameter probability distribution_. As we did for FOSM, let's assume that the bounds in the parameter data section define the range of a Gaussian (or normal) distribution, and that the initial values define the mean. 

### The Prior

We can use `pyemu` to sample parameter values from such a distribution. First, construct a covariance matrix from the parameter data in the `pst` control file:

In [ ]:
v = pyemu.geostats.ExpVario(contribution=1.0,a=2500,anisotropy=1.0,bearing=0.0)
gs = pyemu.utils.geostats.GeoStruct(variograms=[v])
pp_tpl = os.path.join(working_dir,"hkpp.dat.tpl")
cov = pyemu.helpers.geostatistical_prior_builder(pst=pst, struct_dict={gs:pp_tpl})
# display
plt.imshow(cov.to_pearson().x,interpolation="nearest")
plt.colorbar()
cov.to_dataframe().head()

Now re-sample using the geostatistically informed prior:

In [ ]:
parensemble = pyemu.ParameterEnsemble.from_gaussian_draw(pst=pst, cov=cov, num_reals=250,)
# ensure that the samples respect parameter bounds in the pst control file
parensemble.enforce()

Here's an example of the first 5 parameter sets from our draw ("draw" here is like "drawing" a card from a deck):

In [ ]:
parensemble.head()

Now when we plot the spatially distributed parameters (`hk1`) we can see some structure and points which are near to each other are more likely to be similar:

In [ ]:
parnmes = par.loc[par.pargp=='hk1'].parnme.values
pe_k = parensemble.loc[:,parnmes].copy()
# use the hbd convenience function to plot several realisations
hbd.plot_ensemble_arr(pe_k, working_dir, 10)

Let's look at some of the distributions. Note that distributions are log-normal, because parameters in the `pst` are log-transformed:

In [ ]:
for pname in pst.par_names[:5]:
    ax = parensemble.loc[:,pname].hist(bins=20)
    print(parensemble.loc[:,pname].min(),parensemble.loc[:,pname].max(),parensemble.loc[:,pname].mean())
    ax.set_title(pname)
    plt.show()

Notice anything funny? Compare these distributions to the upper/lower bounds in the `pst.parameter_data`. There seem to be many parameters "bunched up" at the bounds. This is due to the gaussian distribution being truncated at the parameter bounds.

In [ ]:
pst.parameter_data.head()

### Run IES

First write the ensemble to an external CSV file and save some options: 1 iteration and no lambda testing

In [ ]:
parensemble.to_csv(os.path.join(working_dir,"prior.csv"))
pst.pestpp_options["ies_par_en"] = "prior.csv"
pst.pestpp_options["ies_lambda_mults"] = [1.0]
pst.pestpp_options["lambda_scale_fac"] = [1.0]
pst.pestpp_options["overdue_giveup_fac"] = 100000

# keep the "truth" values before we repurpose obsval below - the forecast
# plots draw them as a dashed line.
forecast_truth = pst.observation_data.loc[pst.forecast_names,"obsval"].to_dict()
pst.control_data.noptmax = 1
pst.write(os.path.join(working_dir,pst_name),version=2)

As usual, make sure to specify the number of agents to use. This value must be assigned according to the capacity of your machine:

In [ ]:
num_workers = 20

In [ ]:
# the master directory
m_d='master_ies'

Run the next cell to call `pestpp-ies`. It'll take a while.

In [ ]:
pyemu.os_utils.start_workers(working_dir, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            pst_name, # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory
                            )

Alright - let's see some IES results.  For these runs, what was the Phi?
Reload the control file from the master dir to use the result handler:

In [ ]:
pst = pyemu.Pst(os.path.join(m_d,pst_name))

In [ ]:
df_pr = pyemu.ObservationEnsemble(df=pst.ies.obsen0,pst=pst)
df_pt = pyemu.ObservationEnsemble(df=pst.ies.get("obsen",pst.ies.phiactual.iteration.max()),pst=pst)

In [ ]:
df_pr = df_pr.loc[df_pr.phi_vector<1000,:]
df_pt = df_pt.loc[df_pt.phi_vector<1000,:]

In [ ]:
pst.phi

Let's plot Phi for the prior (grey) and posterior (blue) ensembles:

In [ ]:
fig,ax = plt.subplots(1,1)
df_pr.phi_vector.hist(bins=50,ax=ax,fc="0.5",alpha=0.5)
df_pt.phi_vector.hist(bins=50,ax=ax,fc="b",alpha=0.5);


The phi histograms tell us that the posterior ensemble fits better than the prior. The stochastic 1-to-1 plots show us what that means. Every realisation is drawn as a semi-transparent point, so each panel shows the whole *range* of fits the ensemble spans - the ensemble equivalent of the 1-to-1 plots we used for the GLM calibrations. As with the phi histograms above, the **prior is grey** and the **posterior is blue**.

The prior cloud is wide: these are models conditioned on nothing but our expert knowledge. One IES iteration pulls that cloud towards the 1-to-1 line. Crucially though, it does *not* collapse it onto a single point - the spread that remains is our estimate of what the measured data cannot resolve.

In [ ]:
# stochastic 1-to-1: the whole ensemble, prior (grey) and posterior (blue)
hbd.plot_1to1_ensemble(pst, prior=df_pr, posterior=df_pt);

In [ ]:
ptpe = pd.read_csv(os.path.join(m_d,pst_name.replace(".pst",".1.par.csv")),index_col=0)
hbd.plot_ensemble_arr(ptpe.loc[:,parnmes],m_d,10)


## Let's look at the forecasts

In the plots below, prior forecast distributions are shaded grey, posteriors are graded blue and the "true" value is shown as dashed black line.

In [ ]:
for forecast in pst.forecast_names:
    ax = plt.subplot(111)
    df_pr.loc[:,forecast].hist(bins=10,alpha=0.5,color="0.5",ax=ax)
    df_pt.loc[:,forecast].hist(bins=10,alpha=0.5,color="b",ax=ax)
    
    v = forecast_truth[forecast]
    ylim = ax.get_ylim()
    ax.plot([v,v],ylim,"r--",lw=2.0)
    ax.set_ylabel('count')
    ax.set_title(forecast)
    plt.tight_layout()
    plt.show()

Travel time is a hard forecast...

### A smaller ensemble, but more iterations and with lambda testing

In [ ]:
pst.pestpp_options["ies_num_reals"] = 50
pst.pestpp_options["overdue_giveup_fac"] = 100000
pst.pestpp_options.pop("ies_lambda_mults",None)
pst.pestpp_options.pop("lambda_scale_fac",None)
pst.control_data.noptmax = 5


In [ ]:
pst.write(os.path.join(working_dir,pst_name),version=2)

In [ ]:
m_d += "1"

In [ ]:
pyemu.os_utils.start_workers(working_dir, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            pst_name, # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory
                            )

Reload the control file from the master dir to use the result handler:

In [ ]:
pst = pyemu.Pst(os.path.join(m_d,pst_name))
df_pr = pyemu.ObservationEnsemble(df=pst.ies.obsen0,pst=pst)
df_pt = pyemu.ObservationEnsemble(df=pst.ies.get("obsen",pst.ies.phiactual.iteration.max()),pst=pst)

In [ ]:
df_pr = df_pr.loc[df_pr.phi_vector<1000,:]
df_pt = df_pt.loc[df_pt.phi_vector<1000,:]

In [ ]:
fig,ax = plt.subplots(1,1)
df_pr.phi_vector.hist(bins=50,ax=ax,fc="0.5",density=True,alpha=0.5)
df_pt.phi_vector.hist(bins=50,ax=ax,fc="b",density=True,alpha=0.5);

And the stochastic 1-to-1 plots for this smaller ensemble, run for more iterations:

In [ ]:
hbd.plot_1to1_ensemble(pst, prior=df_pr, posterior=df_pt);

We are fitting the data much better - calibration for the win!   right?...right?...right???

In [ ]:
for forecast in pst.forecast_names:
    ax = plt.subplot(111)
    df_pr.loc[:,forecast].hist(bins=10,alpha=0.5,color="0.5",ax=ax)
    df_pt.loc[:,forecast].hist(bins=10,alpha=0.5,color="b",ax=ax)
    v = forecast_truth[forecast]
    ylim = ax.get_ylim()
    ax.plot([v,v],ylim,"r--",lw=2.0)
    ax.set_ylabel('count')
    ax.set_title(forecast)
    plt.tight_layout()
    plt.show()

So with a much better fit, some forecast skill improves, some degrades...A powerful lesson to learn in applied groundwater modeling...

In [ ]:
ptpe = pd.read_csv(os.path.join(m_d,pst_name.replace(".pst",".{0}.par.csv".format(pst.control_data.noptmax))),index_col=0)
hbd.plot_ensemble_arr(ptpe.loc[:,parnmes],m_d,10)